# PM — paso final (PDM)

Toma el Excel de `orquestador_pm.ipynb` (los mercados con `flag_potential = 1`), lo cruza con
ventas mensuales, familias "top 15" compradas y la segmentación estratégica de cada cliente,
y clasifica a cada uno en No Potencial / Low / Medium / High.

Corre **después** de `orquestador_pm.ipynb` — necesita su Excel de salida.

In [ ]:
RUTA_EXCEL_PM = "data_potential/output/20260929_potential_estimations_market_level_mean_seg_pm.xlsx"
RUTA_PDM_1 = "data_potential/input/vs2/pdm_1.csv"
RUTA_PDM_2 = "data_potential/input/vs2/pdm_2.csv"
RUTA_TOP15 = "data_potential/input/vs2/top15.csv"
RUTA_SALIDA = "data_potential/output"

In [ ]:
import sys, time
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.config import get_vertical
from src.pdm import construir_pdm_final, map_segmento_pdm
from src.export import write_excel

cfg = get_vertical("pm_mercado")
factor_umbral = cfg["pdm"]["factor_umbral"]

## Insumos

In [ ]:
t0 = time.time()

potencial_pm = pd.read_excel(RUTA_EXCEL_PM, sheet_name="Potencial", usecols=["mercado_id", "flag_potential"])
potencial_pm = potencial_pm[potencial_pm["flag_potential"] == 1.0]

ids_mercado = pd.read_excel(RUTA_EXCEL_PM, sheet_name="IDs mercado").dropna()

# La hoja "IDs mercado" no trae la segmentación estratégica del cliente, así que se vuelve a
# pedir con la misma query de PM (`fetch_clientes`) — es liviana, no hace falta recorrer todo
# el pipeline de features de nuevo. Se mapea Confitero -> Impulso, sin otra consulta aparte.
from google.cloud import bigquery
from src.data import fetch_clientes

client = bigquery.Client(project="acpe-dev-uc-ml")
clientes_pm = fetch_clientes("pm_mercado", client)
segmento_por_cliente = clientes_pm[["cod_cliente_dex", "des_segmento_estrategico"]].rename(
    columns={"cod_cliente_dex": "Cod Cliente DEX"}
)
segmento_por_cliente["segmento"] = map_segmento_pdm(segmento_por_cliente["des_segmento_estrategico"])
segmento_por_cliente = segmento_por_cliente[["Cod Cliente DEX", "segmento"]]

pdm_1 = pd.read_csv(RUTA_PDM_1)
pdm_2 = pd.read_csv(RUTA_PDM_2)
top15 = pd.read_csv(RUTA_TOP15)
print(f"✓ Insumos listos ({time.time() - t0:.0f}s): pdm_1={len(pdm_1):,}, pdm_2={len(pdm_2):,}, "
      f"top15={len(top15):,}, mercados potenciales={len(potencial_pm):,}")

## Clasificación

In [ ]:
salidas = construir_pdm_final(pdm_1, pdm_2, top15, ids_mercado, segmento_por_cliente, potencial_pm, factor_umbral=factor_umbral)
for hoja, df in salidas.items():
    print(f"{hoja}: {len(df):,} clientes — {df['Potencial'].value_counts().to_dict()}")

## Guardar

In [ ]:
fecha = time.strftime("%Y%m%d")
ruta = Path(RUTA_SALIDA) / f"{fecha}_Potencial_final_PDM.xlsx"
ruta.parent.mkdir(parents=True, exist_ok=True)
write_excel(str(ruta), {"Abarroteros": salidas["Abarroteros"], "Lavanderos": salidas["Lavanderos"],
                        "Impulso": salidas["Impulso"]}, hojas_con_formato=[])
print(f"✓ Guardado en {ruta}")